# Stream quality check

Is the recorded EMG stream good, noisy or disturbed, and by what? This notebook reads a session
folder written by `emgacq` (protocol session, free recording, or a session that crashed) and
checks it in three layers:

1. **Link**: did every sample arrive? Sample rate, lost packets, gaps, reconnections, invalid rows.
2. **Each channel**: electrode contact (offset, drift, pops, railing, flat), interference at rest
   (50 Hz mains, noise floor, low-frequency movement artifacts) and how clearly the EMG of the
   gestures stands out of the rest level.
3. **When**: a second-by-second timeline of what disturbed each channel.

It ends with a verdict (**GOOD** usable as it is, **WARN** usable but look at the flagged items,
**BAD** fix the setup before recording more) and concrete advice for each problem found.

**How to use it.** Open it in VS Code (or Jupyter) with the kernel of the repository's `.venv`,
set `FOLDER` in the next-but-one cell (leave `None` for the latest session) and *Run All*.
Nothing is written to the session folder. The same checks without plots:
`python -m tools.stream_quality [data/<folder>]` (from `DataAquisitionNoCamera/`), or
`--all` for one line per session.

The thresholds are first guesses from the first Cyton sessions of 2026-10-21; they are listed
below and can be overridden per run.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# the notebook lives in DataAquisitionNoCamera/tools: make the emgacq and tools packages importable
here = Path.cwd()
ROOT = next(p for p in [here, *here.parents, here / "DataAquisitionNoCamera"] if (p / "emgacq").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from tools import stream_quality as sq

%matplotlib inline
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

sessions = sq.list_sessions()
pd.DataFrame({"session": [p.name for p in sessions],
              "modified": [pd.Timestamp((p / "emg.csv").stat().st_mtime, unit="s").tz_localize("UTC")
                           .tz_convert("Europe/Rome").strftime("%Y-%m-%d %H:%M") for p in sessions],
              "emg.csv MB": [round((p / "emg.csv").stat().st_size / 1e6, 1) for p in sessions]}).tail(12)

## 1. Session and verdict

`FOLDER`: a folder name inside `data/`, a full path, or `None` for the most recent session.
`THRESHOLDS`: overrides of the defaults shown in the table below (e.g. `{"mains_bad_uv": 80}`).
The printed report is the whole answer in text form; the sections after it show where each
number comes from.

In [ ]:
FOLDER = None
THRESHOLDS = {}

folder = sq.latest_session() if FOLDER is None else (
    Path(FOLDER) if Path(FOLDER).is_absolute() else sq.data_dir() / FOLDER)
rec = sq.load(folder, THRESHOLDS)
rep = sq.analyze(rec)
print(sq.report_text(rep))

In [ ]:
# thresholds in use (WARN above the first value, BAD above the second; SNR: below)
pd.DataFrame([{"setting": k, "value": v} for k, v in rec.th.items()]).set_index("setting").T

## 2. Link

The plot counts the samples that arrived in each second. A healthy Cyton link sits flat on the
nominal 250 samples/s. Dips mean lost packets (red triangles; the package counter of the board
skipped numbers), a drop to zero with hatching is a gap (no data at all, e.g. while
reconnecting). Packet loss below 0.1% is negligible; above 1% the data have holes that matter for
the features. After a reconnection the board restarts: the counter starts again from 0, the
electrode offsets jump, and the first sample can be an invalid all-zero row. This notebook cuts
the stream into **segments** at every gap or invalid row and filters each segment on its own,
so these jumps do not leak into the metrics.

In [ ]:
sq.plot_link(rep)
plt.show()
if rep.link["gaps"]:
    display(pd.DataFrame(rep.link["gaps"]))

## 3. Signals

**Raw** (each segment's median removed): what the amplifier saw. Look for slow drifts and steps
(electrode settling, cables pulled), spikes (pops: an electrode losing contact for an instant)
and traces stuck at a constant value (flat) or at the edge of the range (railed). The Cyton is
DC-coupled, so slow drifts of a few hundred µV are normal; the band-pass removes them.

**Filtered** (20-120 Hz band-pass, 50/100 Hz notched): the EMG band. At rest the trace should
be a thin band of a few µV; during the gestures (coloured bands) it should widen clearly. Cues
are light grey, pauses darker grey, gaps hatched in red. After a reconnection the board
restarts and the baseline needs about 10 s to settle (the exponential drift after the gap).

In [ ]:
sq.plot_signals(rec, "raw")
plt.show()
sq.plot_signals(rec, "emg")
plt.show()

## 4. Noise at rest

At rest the muscles are silent, so whatever is left is noise. Three numbers per channel:

- **Mains (50 Hz)**: the amplitude of the narrow 50 Hz line above the local floor. The mains
  field reaches both electrodes of a pair almost equally, and a bipolar channel subtracts
  them, so it cancels only if the two electrodes have similar contact impedance. A large 50 Hz
  level therefore points to a poor or unequal contact (or a bad BIAS/reference electrode), even
  though the notch filter removes the line itself offline.
- **Noise floor**: RMS in the EMG band with the mains notched. This is the level the EMG has to
  rise above. With good gel electrodes it is around 1-5 µV.
- **Low-frequency (1-15 Hz)**: movement, cable sway, sweat. It sits below the EMG band, but large
  values mean the electrodes or cables move.

In the spectrum, the rest curve (blue) should be low and fairly flat in the shaded EMG band,
with at most a narrow peak at 50 Hz; the gesture curve (orange) should lie well above it across
20-120 Hz. Narrow peaks at other frequencies are listed as "other interference" (switching power
supplies, chargers, displays).

In [ ]:
sq.plot_psd(rep)
plt.show()
display(rep.channels[["channel", "status", "mains_rest_uV", "mains100_rest_uV", "noise_rest_uV",
                      "lowfreq_rest_uV", "dc_mV", "wander_uV", "pops", "railed_pct", "flat_pct"]])
print(f"rest used: {rep.rest_source}, {rep.rest_s} s")
if len(rep.spurs):
    display(rep.spurs.head(12))

## 5. EMG content

The envelope is the moving RMS (250 ms) of the filtered signal: it should rise inside the
gesture bands and fall back in the rests. The bar chart compares, per channel, the RMS at rest
with the RMS of each gesture; the ratio of the strongest gesture to rest is the channel's
signal-to-noise ratio (`snr_best`). Below about 3 the gestures barely show on that channel:
either the electrodes are not over a muscle that works in these gestures, or the contact adds
so much noise that the EMG drowns. A channel can legitimately stay quiet for one gesture (an
extensor during CLOSE), which is why the best gesture is used. A free recording has no labels:
its loudest 10% of seconds stand in for "gestures" and the quietest 20% for rest.

In [ ]:
sq.plot_envelope(rec)
plt.show()
sq.plot_gesture_rms(rep)
plt.show()
display(rep.gesture_rms.round(1))
if rep.trials is not None and len(rep.trials):
    display(rep.trials)

## 6. Correlation between channels at rest

Different electrode pairs over different muscles should be largely independent at rest
(|r| well below 0.5). The most common reason they are not is a common-mode signal: the whole
arm picks up mains and other noise, the BIAS electrode is meant to hold that common level down,
and each bipolar channel cancels what is left only as far as its two electrodes have equal
contact impedance. With a weak BIAS, every unbalanced pair lets part of the same common signal
through, so channels with no electrode in common become correlated; the sign (+ or -) only tells
which electrode of each pair has the higher impedance. Such a pattern changes whenever the
electrodes are re-applied. A pair that correlates in every session, whatever the contact, points
instead to the wiring, or to gel bridging two sites.

In [ ]:
fig = sq.plot_corr(rep)
plt.show() if fig is not None else print("not enough rest data for a correlation")

## 7. Timeline: what disturbs each channel, and when

Every channel is split into 1 s windows, and each window gets the most serious problem it shows
(from worst to mildest: railed, flat, pop, low-freq, mains, noisy; otherwise ok). "noisy" is
judged only in the rest windows, since a high RMS during a gesture is the EMG itself. The top row
shows the protocol. Problems that start at one moment and stay point to an electrode that came
loose or a cable that was moved; problems spread over the whole session point to the setup.

In [ ]:
sq.plot_timeline(rep)
plt.show()
counts = pd.DataFrame({ch: pd.Series(rep.win.cls[:, c]).value_counts(normalize=True) * 100
                       for c, ch in enumerate(rec.channels)}).fillna(0).round(1)
counts.reindex([k for k in sq.WINDOW_CLASSES if k in counts.index])   # % of the windows

## 8. Verdict and what to do

One row per channel with its status and every problem found, then the advice, the most serious
first. The session verdict is BAD when the link is bad or at least half of the channels are bad,
WARN when anything is flagged, GOOD otherwise.

In [ ]:
print(f"VERDICT: {rep.verdict} - {rep.verdict_text}   (link: {rep.link['status']})")
with pd.option_context("display.max_colwidth", None):
    display(rep.channels[["channel", "status", "problems"]])
for grade, where, text in sq.advice(rep):
    print(f"[{grade}] {where}:\n    {text}\n")

## 9. Zoom

Set a time window (seconds from the start of the recording, as on the plots above) to look at
a stretch in detail, e.g. a pop or the moment a channel went noisy. `SCALE_UV` fixes the
vertical scale (`None` = automatic).

In [ ]:
T0, T1 = 20.0, 30.0
SCALE_UV = None

sq.plot_signals(rec, "raw", T0, T1, SCALE_UV)
plt.show()
sq.plot_signals(rec, "emg", T0, T1, SCALE_UV)
plt.show()

## 10. Compare the sessions

One line per session in `data/` (the last 12): verdict, link, and per channel the status with
the rest noise floor (`n`, µV) and the 50 Hz level at rest (`m`, µV). Useful to see whether a
change in the setup (new electrodes, re-prepared skin, laptop on battery) actually helped.

In [ ]:
pd.DataFrame([sq.summary_row(f, THRESHOLDS) for f in sessions[-12:]])